# Closed-loop benchmark smoke

BO Forge v3.3.1 source-only walkthrough. Start Jupyter from the project root or its notebooks directory, including in an extracted source archive. The installed runtime wheel alone does not contain benchmarks.

Run All executes only smoke.yaml: Branin, seed 0, deterministic/noisy modes, and bo/random/sobol strategies. Each of six sequential trials uses four initial observations and six total evaluations, with a 600-second per-trial timeout. Only four BO suggestion calls are needed. The 90-trial standard.yaml specification is deliberately never run here. This is integration evidence, not a claim of BO superiority or measured acceptance. See [Benchmarks](../docs/BENCHMARKS.md).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'benchmarks' / 'specs' / 'smoke.yaml').is_file()
     and (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from a checkout or extracted source archive.')
workspace = TemporaryDirectory(prefix='bo-forge-benchmark-notebook-')
work_dir = Path(workspace.name)
run_dir = work_dir / 'smoke'
report_dir = run_dir / 'report'
regenerated_dir = work_dir / 'regenerated-report'
env = os.environ.copy()
env.pop('PYTHONPATH', None)
env.update({key: '1' for key in (
    'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'
)})
env['MPLBACKEND'] = 'Agg'
print('Temporary run:', run_dir)

In [ ]:
completed = subprocess.run(
    [sys.executable, '-m', 'benchmarks', 'run',
     '--spec', str(PROJECT_ROOT / 'benchmarks' / 'specs' / 'smoke.yaml'),
     '--output', str(run_dir)],
    cwd=PROJECT_ROOT, env=env, capture_output=True, text=True,
    check=False, timeout=3660,
)
print('Runner exit code:', completed.returncode)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if not (run_dir / 'run.json').is_file():
    raise RuntimeError('Run metadata missing; inspect the runner output above.')

## Paired comparisons and noisy versus latent quality

Compare bo/random/sobol within the same problem, mode, and seed. Noisy observed values include observation noise; latent values describe the noise-free objective at the sampled design. A lucky noisy best observation is not necessarily a better latent design. Read latent regret separately from observed best-so-far. One paired seed cannot establish a general strategy ranking.

Quality summaries are conditioned on completed trials. Read planned IDs alongside complete, failed, timeout, interrupted, pending, and running statuses. Partial traces are retained diagnostic evidence, not successful trials, and missing results must not become zero regret. The report is generated automatically even when some trials fail. Summary rows group by problem/mode/strategy and show final regret median and interquartile bounds. Trajectory rows expose contributing_complete, contributing_partial, and scheduled counts; partial traces do not enter completed-trial aggregates. trials.csv records the trial outcomes and denominators.

In [ ]:
display(json.loads((run_dir / 'run.json').read_text(encoding='utf-8')))
statuses = []
for path in sorted((run_dir / 'trials').glob('*/status.json')):
    statuses.append({'trial_directory': path.parent.name,
                     **json.loads(path.read_text(encoding='utf-8'))})
display(pd.DataFrame(statuses))
for filename in ('trials.csv', 'summary.csv', 'trajectories.csv', 'traces.csv'):
    path = report_dir / filename
    if path.is_file():
        print(filename)
        try:
            display(pd.read_csv(path))
        except pd.errors.EmptyDataError:
            print('No rows; inspect trial status and worker.log, not just the summary.')
    else:
        print('Missing report artifact:', filename)
if (report_dir / 'report.md').is_file():
    display(Markdown((report_dir / 'report.md').read_text(encoding='utf-8')))

## Report regeneration and failures

The next command reads the stored run and writes a new report directory. It does not fit models, evaluate objectives, or modify the run directory. Existing report destinations and any trial directory are rejected, preserving source and old report files. Regeneration is not a retry of failed trials. For failures, inspect trials/<id>/status.json, trial.json when present, worker.log, and partial trace.jsonl before drawing conclusions. Missing trial directories must be reconciled against planned IDs in run.json.

In [ ]:
regenerated = subprocess.run(
    [sys.executable, '-m', 'benchmarks', 'report',
     '--run', str(run_dir), '--output', str(regenerated_dir)],
    cwd=PROJECT_ROOT, env=env, capture_output=True, text=True,
    check=False, timeout=120,
)
print(regenerated.stdout)
if regenerated.stderr:
    print(regenerated.stderr)
regenerated.check_returncode()
display(Markdown((regenerated_dir / 'report.md').read_text(encoding='utf-8')))

## Cleanup

This tutorial removes its temporary artifacts after inspection and writes no campaign fixtures or reports into the checkout. For retained acceptance evidence, run the documented CLI separately with a new persistent output directory. If a cell is interrupted or fails, inspect its temporary files before explicitly running cleanup. Keep this notebook output-free in source control.

In [ ]:
workspace.cleanup()
assert not work_dir.exists()

## Optional v3.3.1 routes (outside Run All)

The computational cells above still run only the original six-trial smoke. From the source root, run each schema-v2 route separately with a new output directory:

```bash
python -m benchmarks run --spec benchmarks/specs/mixed_smoke.yaml --output /tmp/bo-forge-benchmark-mixed-smoke
python -m benchmarks run --spec benchmarks/specs/constrained_mixed_smoke.yaml --output /tmp/bo-forge-benchmark-constrained-mixed-smoke
python -m benchmarks run --spec benchmarks/specs/pending_noisy_smoke.yaml --output /tmp/bo-forge-benchmark-pending-noisy-smoke
```

Mixed routes use mixed_quadratic with log_ei; pending_noisy uses Branin with noise_std: 1.0, qlog_nei, and delayed observations through X_pending. Each smoke schedules three strategies at seed 0, four initial and six total evaluations: 9 trials / 54 evaluations across the routes. The opt-in mixed_standard.yaml, constrained_mixed_standard.yaml, and pending_noisy_standard.yaml use seeds 0 through 4 and 24 evaluations (8 initial for mixed routes, 6 for pending): 45 trials / 1,080 evaluations. These are budgets, not guaranteed completions. Local standard acceptance retained 40 complete and five failed constrained-BO trials (1,003 evaluations); no failed outcome was replaced. All nine route-smoke trials completed. Inspect failures, feasibility, pending rows, and unknown timing separately from measured zero. See [Benchmarks](../docs/BENCHMARKS.md) for retained evidence and report regeneration.